In [1]:
!pip install -q transformers accelerate bitsandbytes pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.5 MB/s eta 0:00:00:00:0100:01


In [2]:
!git clone https://github.com/yasdel/Benchmark_RecLLM_Fairness.git

Cloning into 'Benchmark_RecLLM_Fairness'...
remote: Enumerating objects: 167, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 167 (delta 42), reused 20 (delta 20), pack-reused 94 (from 1)
Receiving objects: 100% (167/167), 34.84 MiB | 21.43 MiB/s, done.
Resolving deltas: 100% (78/78), done.


In [3]:
%cd Benchmark_RecLLM_Fairness

/kaggle/working/Benchmark_RecLLM_Fairness


In [4]:
!ls data

df_item_ml-1m.csv
df_user_ml-1m.csv
RecLLM_df_lastFM1k_fullInteraction_80users_nTrack10_lContext5_top10.csv
test_data_lasFM1k_limited_150users.csv
test_data_lastFM1k_fullInteraction_80users.csv
test_data_ml1m_fullInteraction_80users.csv
train_data_lastFM1k_fullInteraction_80users_part_1.csv
train_data_lastFM1k_fullInteraction_80users_part_2.csv
train_data_lastFM1k_fullInteraction_80users_part_3.csv
train_data_lastFM1k_fullInteraction_80users_part_4.csv
train_data_lastFM1k_limitedInteraction_150users_part_1.csv
train_data_lastFM1k_limitedInteraction_150users_part_2.csv
train_data_lastFM1k_limitedInteraction_150users_part_3.csv
train_data_lastFM1k_limitedInteraction_150users_part_4.csv
train_data_ml1m_fullInteraction_80users_part_1.csv
train_data_ml1m_fullInteraction_80users_part_2.csv
train_data_ml1m_fullInteraction_80users_part_3.csv
train_data_ml1m_fullInteraction_80users_part_4.csv
train_prompt_df_lastFM1k_fullInteraction_80users_nTrack10_lContext5.csv
train_prompt_df_lastFM1k_fullIn

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

model.eval()

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLUActivation()
        )
        (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (norm): Phi3RMSNorm((3072,), eps=1e-05)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (lm_head): Linear(in_features=3072, out_featur

In [6]:
def call_llm(prompt):

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return response

In [7]:
import pandas as pd

prompt_df = pd.read_csv(
"data/train_prompt_df_lastFM1k_fullInteraction_80users_nTrack10_lContext5_top_3.csv"
)

print(prompt_df.shape)
prompt_df.head()

(80, 77)


,userId,gender,age_group,itemIds,artistIds,prompt_counterfact-False_sample-random_userDemo-no_info_interact-zero-shot,prompt_counterfact-False_sample-random_userDemo-no_info_interact-ICL-Few-shot-1,prompt_counterfact-False_sample-random_userDemo-no_info_interact-ICL-Few-shot-2,prompt_counterfact-False_sample-random_userDemo-gender_interact-zero-shot,prompt_counterfact-False_sample-random_userDemo-gender_interact-ICL-Few-shot-1,...,prompt_counterfact-True_sample-recent-frequent_userDemo-no_info_interact-ICL-Few-shot-2,prompt_counterfact-True_sample-recent-frequent_userDemo-gender_interact-zero-shot,prompt_counterfact-True_sample-recent-frequent_userDemo-gender_interact-ICL-Few-shot-1,prompt_counterfact-True_sample-recent-frequent_userDemo-gender_interact-ICL-Few-shot-2,prompt_counterfact-True_sample-recent-frequent_userDemo-age-group_interact-zero-shot,prompt_counterfact-True_sample-recent-frequent_userDemo-age-group_interact-ICL-Few-shot-1,prompt_counterfact-True_sample-recent-frequent_userDemo-age-group_interact-ICL-Few-shot-2,prompt_counterfact-True_sample-recent-frequent_userDemo-intersectional_interact-zero-shot,prompt_counterfact-True_sample-recent-frequent_userDemo-intersectional_interact-ICL-Few-shot-1,prompt_counterfact-True_sample-recent-frequent_userDemo-intersectional_interact-ICL-Few-shot-2
0,6,Female,Early Adult (≤24 yrs),"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[813, 8857, 6855, 7218, 2643, 7105, 7728, 4990...",The user is Female and Early Adult (≤24 yrs). ...,The user is Female and Early Adult (≤24 yrs). ...,The user is Female and Early Adult (≤24 yrs). ...,The user is Female and Early Adult (≤24 yrs). ...,The user is Female and Early Adult (≤24 yrs). ...,...,The user is Male and Mid Adult (>24 yrs). The ...,The user is Male and Mid Adult (>24 yrs). The ...,The user is Male and Mid Adult (>24 yrs). The ...,The user is Male and Mid Adult (>24 yrs). The ...,The user is Male and Mid Adult (>24 yrs). The ...,The user is Male and Mid Adult (>24 yrs). The ...,The user is Male and Mid Adult (>24 yrs). The ...,The user is Male and Mid Adult (>24 yrs). The ...,The user is Male and Mid Adult (>24 yrs). The ...,The user is Male and Mid Adult (>24 yrs). The ...
1,46,Female,Early Adult (≤24 yrs),"[13, 74, 77, 155, 195, 203, 204, 217, 341, 345...","[4990, 2187, 6752, 4270, 2247, 2599, 3399, 502...",The user is Female and Early Adult (≤24 yrs). ...,The user is Female and Early Adult (≤24 yrs). ...,The user is Female and Early Adult (≤24 yrs). ...,The user is Female and Early Adult (≤24 yrs). ...,The user is Female and Early Adult (≤24 yrs). ...,...,The user is Male and Mid Adult (>24 yrs). The ...,The user is Male and Mid Adult (>24 yrs). The ...,The user is Male and Mid Adult (>24 yrs). The ...,The user is Male and Mid Adult (>24 yrs). The ...,The user is Male and Mid Adult (>24 yrs). The ...,The user is Male and Mid Adult (>24 yrs). The ...,The user is Male and Mid Adult (>24 yrs). The ...,The user is Male and Mid Adult (>24 yrs). The ...,The user is Male and Mid Adult (>24 yrs). The ...,The user is Male and Mid Adult (>24 yrs). The ...
2,48,Male,Early Adult (≤24 yrs),"[357, 1706, 1707, 1708, 1709, 1710, 1711, 1712...","[5237, 7143, 23731, 9565, 22461, 224, 18603, 3...",The user is Male and Early Adult (≤24 yrs). Th...,The user is Male and Early Adult (≤24 yrs). Th...,The user is Male and Early Adult (≤24 yrs). Th...,The user is Male and Early Adult (≤24 yrs). Th...,The user is Male and Early Adult (≤24 yrs). Th...,...,The user is Female and Mid Adult (>24 yrs). Th...,The user is Female and Mid Adult (>24 yrs). Th...,The user is Female and Mid Adult (>24 yrs). Th...,The user is Female and Mid Adult (>24 yrs). Th...,The user is Female and Mid Adult (>24 yrs). Th...,The user is Female and Mid Adult (>24 yrs). Th...,The user is Female and Mid Adult (>24 yrs). Th...,The user is Female and Mid Adult (>24 yrs). Th...,The user is Female and Mid Adult (>24 yrs). Th...,The user is Female and Mid Adult (>24 yrs). Th...
3,5

In [8]:
from tqdm import tqdm

recommendations_df = pd.DataFrame(index=prompt_df.index)

print("Recommendation dataframe initialized")

Recommendation dataframe initialized


In [9]:
for column in prompt_df.columns[5:]:

    recs = []

    for prompt in tqdm(prompt_df[column]):

        try:
            result = call_llm(prompt)
        except:
            result = None

        recs.append(result)

    recommendations_df[f"recommendation_{column}"] = recs

100%|██████████| 80/80 [04:15<00:00,  3.20s/it]


In [11]:
import os
os.makedirs("results", exist_ok=True)

RecLLM_df = pd.concat([prompt_df, recommendations_df], axis=1)

RecLLM_df.to_csv("results/phi3_experiment_final.csv", index=False)

print("Experiment results saved!")

Experiment results saved!


In [16]:
import pandas as pd
import ast

results = pd.read_csv("results/phi3_experiment_final.csv")

# find prediction columns
pred_cols = [c for c in results.columns if c.startswith("recommendation_")]

print("Recommendation columns:", len(pred_cols))


def normalize_itemlist(x):
    if pd.isna(x):
        return []
    if isinstance(x, str):
        return ast.literal_eval(x)
    return x

results["itemIds"] = results["itemIds"].apply(normalize_itemlist)


for c in pred_cols:

    def hr1(row):
        preds = str(row[c]).split()
        truth = row["itemIds"]
        if preds and truth:
            return int(preds[0] in [str(t) for t in truth])
        return 0

    def hr_all(row):
        preds = str(row[c]).split()
        truth = row["itemIds"]
        if preds and truth:
            hits = sum(1 for p in preds if p in [str(t) for t in truth])
            return hits / len(preds)
        return 0

    results[f"HR@1_{c}"] = results.apply(hr1, axis=1)
    results[f"HR@all_{c}"] = results.apply(hr_all, axis=1)


metrics = results[[c for c in results.columns if c.startswith("HR@")]]

print("\nAverage Metrics:")
print(metrics.mean())

results.to_csv("results/phi3_experiment_scored.csv", index=False)

print("\nEvaluation completed and saved.")

Recommendation columns: 72


/tmp/ipykernel_55/3622450843.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  results[f"HR@1_{c}"] = results.apply(hr1, axis=1)
/tmp/ipykernel_55/3622450843.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  results[f"HR@all_{c}"] = results.apply(hr_all, axis=1)
/tmp/ipykernel_55/3622450843.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-


Average Metrics:
HR@1_recommendation_prompt_counterfact-False_sample-random_userDemo-no_info_interact-zero-shot                          0.000000
HR@all_recommendation_prompt_counterfact-False_sample-random_userDemo-no_info_interact-zero-shot                        0.000126
HR@1_recommendation_prompt_counterfact-False_sample-random_userDemo-no_info_interact-ICL-Few-shot-1                     0.000000
HR@all_recommendation_prompt_counterfact-False_sample-random_userDemo-no_info_interact-ICL-Few-shot-1                   0.000059
HR@1_recommendation_prompt_counterfact-False_sample-random_userDemo-no_info_interact-ICL-Few-shot-2                     0.000000
                                                                                                                          ...   
HR@all_recommendation_prompt_counterfact-True_sample-recent-frequent_userDemo-intersectional_interact-zero-shot         0.000132
HR@1_recommendation_prompt_counterfact-True_sample-recent-frequent_userDemo-int

In [17]:
import pandas as pd

results = pd.read_csv("results/phi3_experiment_scored.csv")

hr1_cols = [c for c in results.columns if c.startswith("HR@1")]
hrall_cols = [c for c in results.columns if c.startswith("HR@all")]

summary = pd.DataFrame({
    "Metric": ["HR@1 Mean", "HR@all Mean"],
    "Value": [results[hr1_cols].mean().mean(), results[hrall_cols].mean().mean()]
})

print(summary)

        Metric     Value
0    HR@1 Mean  0.000000
1  HR@all Mean  0.000097


In [18]:
summary.to_csv("results/phi3_summary_metrics.csv", index=False)

In [19]:
!zip -r results_backup.zip results

  adding: results/ (stored 0%)
  adding: results/tmp.txt (stored 0%)
  adding: results/RecLLM_df_lastFM1k_fullInteraction_80users_nTrack10_lContext5_top10.csv (deflated 92%)
  adding: results/phi3_summary_metrics.csv (deflated 8%)
  adding: results/final_df_foundItemIds_lastFM1k_fullInteraction_80users_nTrack10_lContext5_top3.csv (deflated 74%)
  adding: results/RecLLM_parsed_df_lastFM1k_fullInteraction_80users_nTrack10_lContext5_top3.csv (deflated 72%)
  adding: results/RecLLM_parsed_df_lastFM1k_fullInteraction_80users_nTrack10_lContext5_top10.csv (deflated 76%)
  adding: results/train_prompt_df_lastFM1k_fullInteraction_80users_nTrack10_lContext5_top_3.csv (deflated 93%)
  adding: results/RecLLM_df_lastFM1k_fullInteraction_80users_nTrack10_lContext5_top3.csv (deflated 93%)
  adding: results/RecLLM_parsed_df_lastFM1k_fullInteraction_80users_nTrack10_lContext5.csv (deflated 67%)
  adding: results/phi3_experiment_final.csv (deflated 96%)
  adding: results/final_df_foundItemIds_lastFM1k_f